# Brownian Reading Exercises

This notebook generates reading exercises, defaults set for **6-string electric bass** (low B – high C). Each run produces a fresh set of pieces in MusicXML format, playable and printable from any notation software.

By adjusting `LOW_MIDI`, `RANGE_SEMITONES`, and `TREBLE_THRESHOLD_MIDI` in `generators.py` the same engine works for **any instrument and clef** — 4-string bass, cello, bass clarinet, whatever needs reading practice.

## Algorithmic model

### 1. Pseudo-modal scale
A unique scale is built for each piece via a **strictly upward brownian walk**: starting from a random offset (0–3 semitones above B1), each step adds 1–3 semitones at random until the upper boundary is reached. The result is a one-of-a-kind modal collection covering the full instrument range.

### 2. Melody
A **bidirectional brownian walk** moves over the *indices* of the scale (jumps of ±1–3 positions, with boundary reflection). The resulting 48-note sequence is then **mirrored into a palindrome** (96 notes), giving the exercise a natural arch shape.

### 3. Auto-harmonisation
Every 4/4 measure is automatically harmonised by `harmonizer.getChord`, which analyses the pitched notes in the measure and returns a chord symbol. The symbol is embedded as a chart annotation above the staff.

### 4. Rhythm variants
Three generators share the scale/melody logic but differ in rhythm:

| Generator | Rhythm | Rests |
|---|---|---|
| `gen_quarter` | quarter notes | none |
| `gen_eighth` | eighth notes | ~25 % probability per slot |
| `gen_16th` | mixed 16th / 8th / dotted-8th / quarter | ~25 % probability per slot |

For `gen_16th`, notes or rests that cross a beat boundary are split and tied by the **TECORCO** algorithm.

### 5. Enharmonic notation
Accidentals are assigned **randomly**: black keys may appear as either a sharp or a flat with equal probability (`_rand_enharmonic`). This is intentionally bad notation practice — but a deliberate reading challenge, forcing the player to recognise the same pitch under both spellings.

### 6. Clef handling
The function `_apply_bass_treble_clefs` can insert **treble-8vb clef** changes mid-staff when notes exceed a MIDI threshold, and switch back to **bass clef** when they fall below another threshold (hysteresis). By default both thresholds are set to MIDI 124 (well above the instrument range), so the score stays on bass clef throughout. Lower the thresholds to activate automatic clef changes.

### 7. Collection output
The **N-piece runner** generates `N` independent pieces and concatenates them into a single MusicXML file with page breaks, annotated with the scale and melody range of each piece.


In [ ]:
! pip install music21 showscore

In [ ]:
import sys
sys.path.append('/home/andrea/musica/muProj/comune')
from IPython.display import display, HTML, Javascript
sys.modules['IPython.core.display'] = sys.modules['IPython.display']

import random
from music21 import *
from showscore import show
import harmonizer
import datetime
import math
import tecorco


## Generator functions ##
Three refactored generators — one per process — sharing `_build_scale()` and `_build_mel()`. Each returns a flat `stream.Stream` ready for use standalone or inside the N-piece runner below.

In [ ]:
from generators import *
from generators import _apply_bass_treble_clefs


## N-piece collection runner ##
Set `N` (number of pieces) and `PROCESS` (`'quarter'`, `'16th'`, or `'rests'`) in the cell below. One run produces a single XML file with `N` independently generated pieces separated by page breaks.

In [ ]:

# --- N x PROCESS runner -------------------------------------------
N       = 10
PROCESS = 'eighth'   # 'quarter' | 'eighth' | '16th'
TEMPO   = 60

gen_map = {'quarter': gen_quarter, 'eighth': gen_eighth, '16th': gen_16th}
gen = gen_map[PROCESS]

# Generate raw streams first so scale_range survives before makeMeasures()
raw_pieces   = [gen() for _ in range(N)]
scale_ranges = [getattr(r, 'scale_range', None) for r in raw_pieces]
pieces       = [_apply_bass_treble_clefs(r.makeMeasures()) for r in raw_pieces]

# _piece_range is provided by generators.py
def _piece_range(measured):
    notes = list(measured.recurse().getElementsByClass(note.Note))
    if not notes:
        return None
    midis = [n.pitch.midi for n in notes]
    lo_p = min(notes, key=lambda n: n.pitch.midi).pitch
    hi_p = max(notes, key=lambda n: n.pitch.midi).pitch
    avg_int = round(sum(abs(midis[k+1]-midis[k]) for k in range(len(midis)-1)) / max(len(midis)-1, 1), 1)
    return lo_p.nameWithOctave, hi_p.nameWithOctave, hi_p.midi - lo_p.midi, avg_int

part = stream.Part()
part.insert(0, instrument.Instrument('Violoncello'))

for p_idx, measured in enumerate(pieces):
    rng  = _piece_range(measured)
    srng = scale_ranges[p_idx]
    for m_idx, m in enumerate(measured.getElementsByClass(stream.Measure)):
        if p_idx == 0 and m_idx == 0:
            m.insert(0, tempo.MetronomeMark(number=TEMPO))
        if p_idx > 0 and m_idx == 0:
            m.insert(0, layout.PageLayout(isNew=True))
            m.insert(0, layout.SystemLayout(isNew=True))
        if m_idx == 0 and (rng or srng):
            parts = []
            if srng:
                slo, shi, sspan, savg = srng
                parts.append(f'scale: {slo}–{shi} ({sspan} st, avg step {savg} st)')
            if rng:
                lo, hi, span, avg = rng
                parts.append(f'melody: {lo}–{hi} ({span} st, avg jump {avg} st)')
            te = expressions.TextExpression('  |  '.join(parts))
            te.style.absoluteY = 15
            m.insert(0, te)
        part.append(m)

score = stream.Score()
score.insert(0, metadata.Metadata())
score.metadata.title = f'{N}x {PROCESS} — {datetime.datetime.now():%Y-%m-%d %H:%M}'
score.append(part)

out = f'/home/andrea/musica/scores/6stringExercise/6-string_ex_{PROCESS}_n{N}.xml'
score.write('musicxml', out)
show(score)
